# धडा १३ - काग्नी नॉलेज ग्राफसह एजंट मेमरी


## सेटअप

हे नोटबुक कसे बुद्धिमान **कोडिंग सहाय्यक** तयार करायचे हे दर्शवते जे कायमस्वरूपी स्मृतीसह [**Cognee**](https://www.cognee.ai/) नॉलेज ग्राफ्स आणि **Microsoft Agent Framework** (MAF) वापरते.

Cognee असंरचित मजकूराला संरचित, क्वेरी करण्यायोग्य नॉलेज ग्राफमध्ये रूपांतरित करते जो व्हेक्टर एम्बेडिंगद्वारे समर्थित असतो — तुमच्या एजंटला समृद्ध, नातेसंबंध जाणणारी दीर्घकालीन स्मृती प्रदान करते.

### तुम्हाला काय शिकता येईल
1. **नॉलेज ग्राफ तयार करा**: विकसकांची प्रोफाइल आणि सर्वोत्तम पद्धती संरचित, क्वेरी करण्यायोग्य ज्ञानात रूपांतरित करा.
2. **Cognee चे MAF सोबत एकत्रीकरण**: `@tool` फंक्शन्स वापरून MAF एजंटला Cognee च्या नॉलेज ग्राफशी क्वेरी करण्याची परवानगी द्या.
3. **सेशन-जाणिव असलेली संवाद साधा**: एका सेशनमध्ये अनेक प्रश्नांसाठी संदर्भ राखा.
4. **दीर्घकालीन स्मृती**: महत्त्वाचे ज्ञान सेशन्स दरम्यान जतन करा आणि नवीन संवादांमध्ये ती पुनर्प्राप्त करा.

### पूर्वअटी
- Python 3.9+
- Redis स्थानिकरित्या चालू (`docker run -d -p 6379:6379 redis`) सेशन व्यवस्थापनासाठी
- एक LLM API की (जसे OpenAI) — `.env` मध्ये `LLM_API_KEY` सेट करा
- `.env` मध्ये `CACHING=true` (Cognee सेशन्ससाठी आवश्यक)
- Azure AI Foundry प्रोजेक्ट ज्यात तैनात चॅट मॉडेल आहे
- `.env` मध्ये `AZURE_AI_PROJECT_ENDPOINT` आणि `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI प्रमाणीकरण केलेले (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## एजंट मेमरीचे प्रकार

ही नोटबुक मुख्य धडा १३ नोटबुकमधील तीच तीन मेमरी प्रकार तपासते, परंतु लाँग-टर्म मेमरी बॅकएंड म्हणून Cognee वापरते:

| मेमरी प्रकार | यंत्रणा | आयुष्यमान |
|---|---|---|
| **वर्किंग** | `agent.create_session()` (MAF) | एकच संभाषण |
| **शॉर्ट-टर्म** | Cognee सेशन कॅश (Redis) | एकच सेशन |
| **लाँग-टर्म** | Cognee ज्ञान ग्राफ + व्हेक्टर | कायमस्वरूपी |

### Cognee ची मेमरी आर्किटेक्चर
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## कॉग्नी स्टोरेज तयार करा


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Part 1 — ज्ञानाधारित बांधणी

आमच्या कोडिंग सहाय्यकासाठी एक व्यापक ज्ञानाधारित तयार करण्यासाठी आम्ही तीन प्रकारचे डेटा इन्गेस्ट करतो:

1. **डेव्हलपर प्रोफाइल** — वैयक्तिक कौशल्य आणि तांत्रिक पार्श्वभूमी  
2. **पायथन सर्वोत्तम पद्धती** — पायथनचे झेन प्रायोगिक मार्गदर्शकांसह  
3. **ऐतिहासिक संभाषणे** — डेवलपर्स आणि AI सहाय्यकांमधील मागील प्रश्नोत्तर सत्र


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## ज्ञान ग्राफ दिसवा

Cognee त्या घटकांची आणि संबंधांची एक इंटरऐक्टिव HTML दृष्यरूप तयार करू शकतो जे त्याने काढले आहेत.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## मेमरीला मेमिफायने समृद्ध करा

`memify()` ज्ञान ग्राफचे विश्लेषण करते आणि बुद्धिमान नियम तयार करते — नमुने, सर्वोत्तम सराव आणि संकल्पनांमधील संबंधित गोष्टी ओळखते.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## भाग 2 — कॉगनी टूल्ससह MAF एजंट

आता आपण एक MAF एजंट तयार करतो जो `@tool` फंक्शन्सद्वारे कॉगनीच्या ज्ञान ग्राफवर क्वेरी करू शकतो. हे एजंटला ग्राफ-जागरूक सेमांटिक शोधाची पूर्ण क्षमता वापरण्याची परवानगी देते, तर सत्रांमधून संभाषणात्मक संदर्भ राखतो.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## सत्रांसह कार्यकारी स्मृती

`AgentSession` (जे `agent.create_session()` द्वारे तयार केले जाते) सत्रात कार्यकारी स्मृती प्रदान करते. एजंट पूर्वीच्या संदेशांकडे लक्ष देऊ शकतो आणि एकाच वेळी Cognee च्या दीर्घकालीन ज्ञान ग्राफचाही शोध घेऊ शकतो.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## नवीन सत्र — दीर्घकालीन स्मृती टिकून राहते

नवीन सत्र सुरू केल्याने वर्किंग मेमरी साफ होते, परंतु Cognee ज्ञान ग्राफ अजूनही उपलब्ध असतो. एजंट पूर्णपणे नवीन संभाषणात देखील तेच दीर्घकालीन ज्ञान पुनर्प्राप्त करू शकतो.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

या नोटबुकमध्ये तुम्ही एक कोडिंग सहायक तयार केला ज्यामध्ये **MAF चे कार्यशील स्मृती** (`agent.create_session()`) आणि **Cognee चे दीर्घकालीन ज्ञान ग्राफ** यांचा समावेश आहे.

### तुम्ही काय शिकलात
1. **ज्ञान ग्राफ तयार करणे**: Cognee अप्रतिबंधित मजकूर स्वीकारते आणि ग्राफ + व्हेक्टर स्मृती तयार करते.
2. **Graph enrichment with memify**: तुमच्या विद्यमान ग्राफवरून व्युत्पन्न तथ्ये आणि समृद्ध संबंधित नातेसंबंध तयार केले.
3. **MAF + Cognee समाकलन**: `@tool` फंक्शन्स MAF एजंटला Cognee च्या ग्राफवर नैसर्गिकपणे क्वेरी करण्याची परवानगी देतात.
4. **कार्यशील स्मृती + दीर्घकालीन स्मृती**: `AgentSession` (द्वारे `agent.create_session()`) सत्र संदर्भ प्रदान करतो, तर Cognee कायमस्वरूपी ज्ञान पुरवते.
5. **NodeSets सह फिल्टर केलेले शोध**: ज्ञान ग्राफच्या विशिष्ट उपसमुच्चयांवर लक्ष ठेवा (उदा. फक्त तत्त्वे).

### मुख्य मुद्दे
- **Cognee** अनस्ट्रक्चर्ड मजकूराला संरचित, नातेसंबंध-अनुभवी स्मृतीमध्ये रूपांतरित करतो — ज्यामुळे सरळ व्हेक्टर साठवणूकपेक्षा अधिक सामर्थ्यवान होते.
- **`@tool` functions** MAF एजंट्स आणि बाह्य ज्ञान प्रणालींमध्ये स्वच्छ सेतू निर्माण करतात.
- **`AgentSession`** (द्वारे `agent.create_session()`) प्रति संभाषण संदर्भ दीर्घकाळ टिकणाऱ्या ज्ञानापासून वेगळा ठेवतो.
- एकाच ज्ञान ग्राफ विविध सत्रे आणि एजंटसाठी वापरला जातो.

### वास्तविक जगातील अनुप्रयोग
- **डेव्हलपर कोपिलॉट्स**: कोड पुनरावलोकन, घटनेचे विश्लेषण, आर्किटेक्चर सहाय्यक
- **ग्राहकांसमोरील कोपिलॉट्स**: उत्पादन कागदपत्रे, FAQ, आणि CRM नोट्सवर आधारित समर्थन एजंट्स
- **आंतररिक तज्ञ कोपिलॉट्स**: धोरणे, कायदेशीर किंवा सुरक्षा सहाय्यक, मार्गदर्शक तत्वांवर आधारित कारणे
- **एकसंध डेटा स्तर**: संरचित आणि अप्रतिबंधित डेटा एका क्वेरीयोग्य ग्राफमध्ये एकत्र करा

### पुढील पावले
- Cognee मध्ये कालिक जागरूकतेची चाचणी करा
- विशिष्ट क्षेत्रासाठी OWL ओन्टोलॉजी व्याख्यित करा जे ग्राफच्या गुणवत्तेवर लक्ष ठेवेल
- युजर फीडबॅक लूप्स जोडून वेळोवेळी पुनर्प्राप्ती सुधारित करा
- एकाच Cognee स्मृती स्तर सामायिक करणाऱ्या बहु-एजंट प्रणालींवर स्केल करा


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**सूचना**:
हा दस्तऐवज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित करण्यात आला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात ठेवा की स्वयंचलित अनुवादांमध्ये चुका किंवा अनुचितता असू शकतात. मूळ दस्तऐवज त्याच्या मूळ भाषेतच अधिकारप्राप्त स्रोत मानला पाहिजे. महत्त्वाच्या माहितीकरिता, व्यावसायिक मानवी अनुवादाची शिफारस केली जाते. या अनुवादाच्या वापरामुळे उद्भवणार्‍या कोणत्याही गैरसमज वा चुकीच्या अर्थनिर्णयाबद्दल आम्ही जबाबदार नाहीत.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
